# PRF Multas — notebook completo do pipeline (2007–2025)

**92.215.588 multas de trânsito da PRF**, 19 anos unificados, normalizados e tratados em uma
**base analítica de 28 colunas** (Parquet).

Este notebook reúne **todos os scripts usados até agora** (as células `%%writefile` recriam
os arquivos em `scripts/` com o conteúdo exato do repositório), a **implementação de colunas**,
o histórico de **alterações** e **células de validação** que rodam em segundos sobre os parquets.

## Índice
1. Configuração
2. **Fase 1 — ingestão**: `extract_rar.js` → `01_unificar_anos` (2007–2017) → `02_extrair_unificar` (2018–2025) → `03_normalizar` (schema de 25 colunas)
3. **Fase 2 — tratamento**: `04_tratar` → base analítica de **28 colunas** (`MES` nome do mês, `ANO`, `QUARTIL` Q1–Q4)
4. Validações executáveis (contagens, `MES`/`QUARTIL`, tipos, nulos)
5. Histórico de alterações

> **Aviso:** as células `%%writefile` sobrescrevem `scripts/*` com o mesmo conteúdo (idempotente).
> As etapas pesadas (horas de execução) estão documentadas, **não** re-executadas por padrão —
> `EXECUTAR_PIPELINE = False` mantém o notebook rápido.

## 1. Configuração

Rode a partir da raiz do repo **ou** de `notebooks/` — a célula ajusta o diretório sozinha.
O único requisito é `duckdb` (`requirements.txt`: `duckdb==2.0.1`, `.venv`).

In [1]:
# Configuração: roda da raiz ou de notebooks/
from pathlib import Path
import os
import duckdb

RAIZ = Path.cwd()
if not (RAIZ / "scripts").exists() and (RAIZ.parent / "scripts").exists():
    RAIZ = RAIZ.parent
os.chdir(RAIZ)
print("RAIZ =", RAIZ)

EXECUTAR_PIPELINE = False   # True = reexecuta etapas pesadas (horas); False = só análises

con = duckdb.connect()
con.execute("SET threads=2")
con.execute("SET memory_limit='2400MB'")

RAIZ = /home/user/PRF-MULTAS-PROJETO


## 2. Fase 1 — ingestão (scripts)

**Fontes:** Google Drive `PRF_MULTAS` (11 RARs bimestrais/mensais 2007–2017) e
`Multas_Part` (21 ZIPs mensais 2018–2025, todos < 100 MB — regra do usuário).

Fluxo: baixar → extrair → unificar **um CSV por ano** → normalizar para o **schema canônico de 25 colunas**.

| Script | Etapa |
|--------|-------|
| `extract_rar.js` | extrai RARs (node + node-unrar-js), achata nomes |
| `01_unificar_anos.py` | 2007–2017: 11 anos, 39.696.981 linhas |
| `01b_montar_partes.py` | histórico (superseded — montagem de partes) |
| `02_extrair_unificar.py` | 2018–2025: 8 anos, 52.518.607 linhas + saneamento de aspas |
| `03_normalizar.py` | normalização → 25 colunas, 19/19 anos |

### 2.1 Extração dos RARs (2007–2017)

`node scripts/extract_rar.js <rar> <outDir>` — detecta 6 arquivos bimestrais (2007–2014)
ou 12 mensais (2015–2017) por ano.

In [2]:
%%writefile scripts/extract_rar.js
// Extrai um .rar para um diretório usando node-unrar-js (WASM, sem binário do sistema)
const { createExtractorFromData } = require('node-unrar-js');
const fs = require('fs');
const path = require('path');
(async () => {
  const [rarPath, outDir] = process.argv.slice(2);
  const data = fs.readFileSync(rarPath);
  const ex = await createExtractorFromData({ data });
  fs.mkdirSync(outDir, { recursive: true });
  let n = 0, skipped = 0;
  for (const f of ex.extract().files) {
    const name = f.fileHeader.name;
    if (name.endsWith('/') || f.fileHeader.packSize === 0) { skipped++; continue; }
    if (!f.extraction) { skipped++; continue; }
    const dest = path.join(outDir, path.basename(name));
    fs.writeFileSync(dest, f.extraction);
    n++;
  }
  console.log(`OK ${path.basename(rarPath)}: ${n} arquivos extraidos, ${skipped} entradas ignoradas -> ${outDir}`);
})().catch(e => { console.error('FALHA:', e.message); process.exit(1); });

Overwriting scripts/extract_rar.js


### 2.2 Unificação 2007–2017 (`01`)

Lê `google_drive/_extracted/<ano>/`, detecta delimitador/encoding por amostragem,
e escreve `dados_brutos/infracoes_<ano>.csv` com contagem validada (entrada == saída).
**Resultado: 39.696.981 linhas.**

In [3]:
%%writefile scripts/01_unificar_anos.py
#!/usr/bin/env python3
"""
01 — Ingestão: unifica os CSVs mensais/bimestrais de cada ano
em UM arquivo por ano (dados_brutos/infracoes_<ANO>.csv).

Motor: DuckDB (import duckdb) — leitura e escrita via SQL.
Regras:
  - Ordem cronológica explícita dos arquivos (jan..dez / bimestres).
  - all_varchar=true: só une, não infere tipos (tratamento vem depois).
  - Validação: contagem de linhas de entrada == saída.
  - Cabeçalho idêntico dentro do ano (assert).
"""
import hashlib
import sys
import time
from pathlib import Path

import duckdb

EXTRACTED = Path("google_drive/_extracted")
OUT_DIR = Path("dados_brutos")
OUT_DIR.mkdir(exist_ok=True)

MESES = ["jan", "fev", "mar", "abr", "mai", "jun",
         "jul", "ago", "set", "out", "nov", "dez"]
BIMESTRES = ["jan_fev", "mar_abr", "mai_jun", "jul_ago", "set_out", "nov_dez"]


def ordenar_arquivos(year_dir: Path) -> list[Path]:
    stems = {p.stem for p in year_dir.glob("*.csv")}
    if len(stems) == 12 and all(m in stems for m in MESES):
        return [year_dir / f"{m}.csv" for m in MESES]
    if len(stems) == 6 and all(b in stems for b in BIMESTRES):
        return [year_dir / f"{b}.csv" for b in BIMESTRES]
    raise SystemExit(f"[ERRO] Padrão de arquivos não reconhecido em {year_dir}: {sorted(stems)}")


def detectar_encoding(files: list[Path]) -> str:
    """Detecta a encoding do ano: utf-8 se todos decodificam, senao latin-1."""
    for f in files:
        raw = f.read_bytes()
        try:
            raw.decode("utf-8")
        except UnicodeDecodeError:
            return "latin-1"
    return "utf-8"


def header_hash(path: Path) -> str:
    with open(path, "rb") as fh:
        return hashlib.md5(fh.readline()).hexdigest()


def unificar_ano(year: str) -> dict:
    year_dir = EXTRACTED / year
    files = ordenar_arquivos(year_dir)

    # assert: cabeçalho idêntico em todos os arquivos do ano
    hashes = {header_hash(f) for f in files}
    if len(hashes) != 1:
        raise SystemExit(f"[ERRO] Cabeçalhos divergentes em {year}: {hashes}")

    enc = detectar_encoding(files)
    out = OUT_DIR / f"infracoes_{year}.csv"
    con = duckdb.connect()
    try:
        file_list = ", ".join(f"'{f.resolve()}'" for f in files)
        # contagem de entrada (registros, sem cabeçalho)
        n_in = con.execute(
            f"SELECT count(*) FROM read_csv([{file_list}], "
            f"header=true, delim=',', all_varchar=true, encoding='{enc}')"
        ).fetchone()[0]

        t0 = time.time()
        con.execute(
            f"COPY (SELECT * FROM read_csv([{file_list}], "
            f"header=true, delim=',', all_varchar=true, encoding='{enc}')) "
            f"TO '{out.resolve()}' (HEADER, DELIMITER ',', QUOTE '\"')"
        )
        dt = time.time() - t0

        n_out = con.execute(
            f"SELECT count(*) FROM read_csv('{out.resolve()}', "
            "header=true, delim=',', all_varchar=true)"
        ).fetchone()[0]

        if n_in != n_out:
            raise SystemExit(f"[ERRO] {year}: entrada={n_in} != saida={n_out}")

        size_mb = out.stat().st_size / 1e6
        print(f"OK {year}: {n_out:,} registros | {len(files)} arquivos -> "
              f"{out.name} ({size_mb:.1f} MB, {dt:.1f}s)")
        return {"ano": year, "registros": n_out, "arquivos": len(files),
                "mb": round(size_mb, 1), "segundos": round(dt, 1)}
    finally:
        con.close()


def main(anos: list[str]):
    resumo = []
    for year in anos:
        resumo.append(unificar_ano(year))
    print("\n=== RESUMO ===")
    total = sum(r["registros"] for r in resumo)
    for r in resumo:
        print(f"  {r['ano']}: {r['registros']:>10,} registros  ({r['mb']} MB)")
    print(f"  TOTAL: {total:,} registros")


if __name__ == "__main__":
    if len(sys.argv) > 1:
        anos = sys.argv[1:]
    else:
        anos = sorted(d.name for d in EXTRACTED.iterdir() if d.is_dir())
    main(anos)

Overwriting scripts/01_unificar_anos.py


### 2.3 (histórico) Montagem de partes — `01b`

Primeira abordagem de particionamento dos ZIPs grandes; **superseded** pelo fluxo definitivo
(`02` + ZIPs < 100 MB). Mantido apenas como registro.

In [4]:
%%writefile scripts/01b_montar_partes.py
#!/usr/bin/env python3
"""
01b — Monta os ZIPs de 2018–2025 a partir de volumes < 100 MB vindos do Drive.

Os volumes chegam em google_drive/ com nomes do tipo:
  infracoes_2018.zip.001, .002, ...   (7-Zip "Dividir arquivo")
  infracoes_2018.z01, .z02, ..., .zip  (WinZip/arrasto de volumes)
  infracoes_2018_part_1, _part_2, ...

Este script:
  1. Agrupa os volumes por nome-base.
  2. Concatena em ordem numérica → arquivo .zip único (remontagem byte a byte).
  3. Valida o ZIP (zipfile.testzip).
  4. Extrai para google_drive/_extracted/<ANO>/ (achatando subpastas).

Depois rode a unificação normal:
  .venv/bin/python scripts/01_unificar_anos.py <anos...>
"""
import re
import sys
import zipfile
from pathlib import Path

DRIVE = Path("google_drive")
EXTRACTED = DRIVE / "_extracted"


def agrupar_partes(files: list[Path]) -> dict[str, list[tuple[int, Path]]]:
    grupos: dict[str, list[tuple[int, Path]]] = {}
    for f in files:
        nome = f.name
        m = re.match(r"^(?P<base>.+)\.(?P<num>\d{3,})$", nome)          # x.zip.001 / x.001
        if m:
            grupos.setdefault(m.group("base"), []).append((int(m.group("num")), f))
            continue
        m = re.match(r"^(?P<base>.+)\.z(?P<num>\d{2,})$", nome, re.I)   # x.z01
        if m:
            grupos.setdefault(m.group("base"), []).append((int(m.group("num")), f))
            continue
        m = re.match(r"^(?P<base>.+?)[._-]part[._-]?(?P<num>\d+)$", nome, re.I)  # x_part_1
        if m:
            grupos.setdefault(m.group("base"), []).append((int(m.group("num")), f))
    return grupos


def ano_de(base: str) -> str | None:
    m = re.search(r"20\d{2}", base)
    return m.group(0) if m else None


def montar(grupo_base: str, partes: list[tuple[int, Path]]) -> Path:
    partes.sort(key=lambda t: t[0])
    nums = [n for n, _ in partes]
    if nums != list(range(nums[0], nums[0] + len(nums))):
        print(f"  [AVISO] sequência de volumes com lacunas: {nums}")

    destino = DRIVE / (grupo_base if grupo_base.endswith(".zip") else grupo_base + ".zip")
    with open(destino, "wb") as out:
        for _, p in partes:
            out.write(p.read_bytes())
    print(f"  remontado: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB, "
          f"{len(partes)} volumes)")
    return destino


def extrair(zip_path: Path, ano: str) -> None:
    destino = EXTRACTED / ano
    destino.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad:
            raise SystemExit(f"[ERRO] ZIP corrompido em {zip_path.name}: {bad}")
        for info in zf.infolist():
            if info.is_dir():
                continue
            nome = Path(info.filename).name          # achata subpastas
            if not nome.lower().endswith(".csv"):
                print(f"  (ignorado não-CSV: {info.filename})")
                continue
            with zf.open(info) as src, open(destino / nome, "wb") as dst:
                dst.write(src.read())
    csvs = sorted(p.name for p in destino.glob("*.csv"))
    print(f"  extraído: {zip_path.name} -> _extracted/{ano}/ ({len(csvs)} CSVs: {csvs})")


def main(alvos: list[str]):
    files = [f for f in DRIVE.iterdir() if f.is_file()]
    grupos = agrupar_partes(files)
    if not grupos:
        raise SystemExit("[ERRO] Nenhum conjunto de volumes encontrado em google_drive/. "
                         "Esperado: .001/.002, .z01/.z02 ou _part_N.")

    for base, partes in sorted(grupos.items()):
        ano = ano_de(base)
        if alvos and ano not in alvos:
            continue
        if ano is None:
            print(f"[AVISO] sem ano no nome: {base} — pulando")
            continue
        print(f"[{ano}] {base}: {len(partes)} volumes")
        z = montar(base, partes)
        extrair(z, ano)


if __name__ == "__main__":
    main(sys.argv[1:])

Overwriting scripts/01b_montar_partes.py


### 2.4 Unificação 2018–2025 (`02`)

- codecs (UTF-8/latin-1) + transcodificação; `sanear_aspas` corrige linhas com aspas
  quebradas (ex.: `infrações2024_12b.csv` linha 195138);
- checagem de nº de campos por linha e conferência `count(*)` no DuckDB;
- 2020 usa `union_by_name` (divergência de schema de jan/2020);
- **`MESES_INEXISTENTES = {2024: [6]}`** — jun/2024 **não existe** na base
  (direção do usuário: considerar inexistente, logado como `[INFO]`, nunca como pendência);
- ZIPs de origem removidos após validação (libera disco).
**Resultado: 52.518.607 linhas.**

In [5]:
%%writefile scripts/02_extrair_unificar.py
#!/usr/bin/env python3
"""
02 — Ingestão 2018–2025: extrai os CSVs mensais dos ZIPs particionados
(pasta Drive Multas_Part), transcodifica para UTF-8 e unifica em UM
arquivo por ano em dados_brutos/infracoes_<ANO>.csv.

Regras:
  - Ordem cronológica de meses (inclui sufixos 12a/12b de 2024).
  - MESES_INEXISTENTES: meses que não existem na fonte (ex.: jun/2024)
    são ignorados e não sinalizam o ano como incompleto.
  - Detecção de encoding por arquivo: utf-8 | cp1252 | latin-1 -> UTF-8.
  - Cabeçalho idêntico dentro do ano (assert), exceto 2020:
    jan/2020 tem coluna extra ("UF Placa", 23 vs 22 colunas) ->
    unificação via DuckDB union_by_name=true.
  - Delimitadores: 2018 = ',' ; 2019–2025 = ';'.
  - Validação obrigatória: soma das contagens por mês (DuckDB) ==
    contagem do arquivo final; colunas conferidas via DESCRIBE.
  - Apaga os ZIPs do ano e os temporários SOMENTE após validação.

Uso: .venv/bin/python scripts/02_extrair_unificar.py [ANO ...]
"""
import codecs
import re
import shutil
import sys
import time
import zipfile
from pathlib import Path

import duckdb

DRIVE = Path("google_drive")
TMP = DRIVE / "_tmp"
OUT_DIR = Path("dados_brutos")
OUT_DIR.mkdir(exist_ok=True)

MESES = {"jan": 1, "fev": 2, "mar": 3, "abr": 4, "mai": 5, "jun": 6,
         "jul": 7, "ago": 8, "set": 9, "out": 10, "nov": 11, "dez": 12}

DELIM = {2018: ","}
DELIM_PADRAO = ";"

# Meses que NÃO EXISTEM na base original (confirmado pelo usuário) — não são
# pendência de ingestão e não devem sinalizar o ano como incompleto.
MESES_INEXISTENTES: dict[int, list[int]] = {
    2024: [6],   # junho/2024 não existe na fonte
}


# ---------- codecs ----------

def detectar_codec(path: Path) -> str:
    """utf-8 (com ou sem BOM) | cp1252 | latin-1 — por arquivo, streaming."""
    try:
        dec = codecs.getincrementaldecoder("utf-8")()
        with open(path, "rb") as f:
            while True:
                chunk = f.read(1 << 20)
                if not chunk:
                    break
                dec.decode(chunk)
            dec.decode(b"", final=True)
        return "utf-8"
    except UnicodeDecodeError:
        pass
    try:
        with open(path, "rb") as f:
            while True:
                chunk = f.read(1 << 20)
                if not chunk:
                    break
                chunk.decode("cp1252")
        return "cp1252"
    except UnicodeDecodeError:
        return "latin-1"


def transcodificar(path: Path, codec: str) -> None:
    """Reescreve o arquivo em UTF-8 (no-place para cp1252/latin-1)."""
    if codec == "utf-8":
        return
    tmp = path.with_suffix(".utf8.tmp")
    with open(path, "rb") as fin, open(tmp, "wb") as fout:
        while True:
            chunk = fin.read(1 << 20)
            if not chunk:
                break
            fout.write(chunk.decode(codec).encode("utf-8"))
    tmp.replace(path)


def ler_header(path: Path) -> bytes:
    """Primeira linha (sem BOM, sem CR/LF final)."""
    with open(path, "rb") as f:
        line = f.readline()
    if line.startswith(codecs.BOM_UTF8):
        line = line[len(codecs.BOM_UTF8):]
    return line.rstrip(b"\r\n")


def sanear_aspas(path: Path) -> int:
    """Corrige linhas com número ÍMPAR de aspas (CSV malformado na origem).

    Padrão observado (unico na base): campo de modelo sem aspa de fechamento
    antes da quilometragem, ex.:
      ...;"VERSA 16 UNIQUE XXX;58197;"Desc...
    ->  ...;"VERSA 16 UNIQUE XXX";58197;"Desc...
    Retorna o nº de linhas corrigidas; aborta se nao convergir para par.
    """
    import re
    padrao = re.compile(rb'"([^";]+);(\d+);"')
    corrigidas = 0
    tmp = path.with_suffix(".fix.tmp")
    with open(path, "rb") as fin, open(tmp, "wb") as fout:
        for linha in fin:
            if linha.count(b'"') % 2 == 1:
                nova = padrao.sub(rb'"\1";\2;"', linha)
                if nova.count(b'"') % 2 == 1:
                    raise SystemExit(
                        f"[ERRO] linha com aspas ímpares irreparável em "
                        f"{path.name}: {linha[:300]!r}"
                    )
                fout.write(nova)
                corrigidas += 1
            else:
                fout.write(linha)
    tmp.replace(path)
    if corrigidas:
        print(f"  [SANIDADE] {path.name}: {corrigidas} linha(s) corrigida(s) de aspas ímpares")
    return corrigidas


# ---------- mapeamento zip -> meses ----------

def coletar_meses(ano: str) -> list[tuple[tuple[int, str], Path, str]]:
    """[(ordem, zip, member)] ordenado cronologicamente para o ano."""
    itens = []
    for z in sorted(DRIVE.glob("*.zip")):
        if not re.search(ano, z.name):
            continue
        with zipfile.ZipFile(z) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                nome = Path(info.filename).name.lower()
                m = re.search(r"_(\d{2})([a-z]?)\.csv$", nome)
                if m:
                    chave = (int(m.group(1)), m.group(2))
                elif nome[:-4] in MESES:
                    chave = (MESES[nome[:-4]], "")
                else:
                    raise SystemExit(f"[ERRO] nome não reconhecido: {info.filename} ({z.name})")
                itens.append((chave, z, info.filename))
    itens.sort(key=lambda t: t[0])
    return itens


def extrair(zip_path: Path, member: str, destino: Path) -> None:
    with zipfile.ZipFile(zip_path) as zf, zf.open(member) as src, open(destino, "wb") as dst:
        shutil.copyfileobj(src, dst, 1 << 20)


# ---------- unificação ----------

def contar_duckdb(path: Path, delim: str) -> int:
    con = duckdb.connect()
    try:
        return con.execute(
            f"SELECT count(*) FROM read_csv('{path.resolve()}', header=true, "
            f"delim='{delim}', quote='\"', escape='\"', all_varchar=true, "
            "encoding='utf-8', strict_mode=false, parallel=false)"
        ).fetchone()[0]
    finally:
        con.close()


def garantir_newline(f):
    if f.tell() == 0:
        return
    f.seek(-1, 2)
    if f.read(1) != b"\n":
        f.seek(0, 2)
        f.write(b"\n")
    else:
        f.seek(0, 2)


def unificar_uniforme(ano: str, itens, delim: str, tmp_dir: Path) -> dict:
    """Concatenação com cabeçalho único (meses com cabeçalho idêntico)."""
    out = OUT_DIR / f"infracoes_{ano}.csv"
    meses_ok, linhas = [], 0
    header_base = None
    with open(out, "w+b") as fo:
        for idx, ((mes, suf), zpath, member) in enumerate(itens):
            tmp = tmp_dir / f"{ano}_{mes:02d}{suf}.csv"
            extrair(zpath, member, tmp)
            codec = detectar_codec(tmp)
            transcodificar(tmp, codec)
            sanear_aspas(tmp)
            hdr = ler_header(tmp)
            if idx == 0:
                header_base = hdr
                fo.write(hdr + b"\n")
            elif hdr != header_base:
                raise SystemExit(
                    f"[ERRO {ano}] cabeçalho divergente em mês {mes:02d}{suf}:\n"
                    f"  base: {header_base[:200]!r}\n"
                    f"  este: {hdr[:200]!r}"
                )
            n = contar_duckdb(tmp, delim)
            linhas += n
            with open(tmp, "rb") as fi:
                fi.readline()               # consome o cabeçalho
                shutil.copyfileobj(fi, fo, 1 << 20)
            garantir_newline(fo)
            meses_ok.append(f"{mes:02d}{suf}")
            tmp.unlink()
        n_cols = header_base.count(b";" if delim == ";" else b",") + 1

    n_out = contar_duckdb(out, delim)
    if n_out != linhas:
        raise SystemExit(f"[ERRO {ano}] entrada={linhas} != saida={n_out}")
    return {"arquivo": out, "registros": n_out, "meses": meses_ok,
            "cols": n_cols, "delim": delim}


def unificar_2020(itens, tmp_dir: Path) -> dict:
    """jan/2020 com coluna extra -> DuckDB union_by_name."""
    paths, headers = [], {}
    for (mes, suf), zpath, member in itens:
        tmp = tmp_dir / f"2020_{mes:02d}{suf}.csv"
        extrair(zpath, member, tmp)
        codec = detectar_codec(tmp)
        transcodificar(tmp, codec)
        sanear_aspas(tmp)
        headers[f"{mes:02d}{suf}"] = ler_header(tmp).decode("utf-8", "replace")
        paths.append(tmp)

    distintos = {}
    for k, h in headers.items():
        distintos.setdefault(h, []).append(k)
    print(f"  [2020] headers distintos: {len(distintos)}")
    if len(distintos) > 1:
        grupos = list(distintos.items())
        base_cols = grupos[0][0].split(";")
        for h, meses in grupos[1:]:
            outras = h.split(";")
            print(f"  [2020] {meses} vs {grupos[0][1]}: "
                  f"só_em_{meses[0]}={[c for c in outras if c not in base_cols]} | "
                  f"só_em_{grupos[0][1][0]}={[c for c in base_cols if c not in outras]}")

    linhas = sum(contar_duckdb(p, ";") for p in paths)
    out = OUT_DIR / "infracoes_2020.csv"
    con = duckdb.connect()
    try:
        file_list = ", ".join(f"'{p.resolve()}'" for p in paths)
        con.execute(
            f"COPY (SELECT * FROM read_csv([{file_list}], header=true, delim=';', "
            "quote='\"', escape='\"', all_varchar=true, encoding='utf-8', "
            "union_by_name=true, strict_mode=false)) "
            f"TO '{out.resolve()}' (HEADER, DELIMITER ';', QUOTE '\"')"
        )
        n_out = con.execute(
            f"SELECT count(*) FROM read_csv('{out.resolve()}', header=true, "
            "delim=';', quote='\"', escape='\"', all_varchar=true, "
            "encoding='utf-8', strict_mode=false)"
        ).fetchone()[0]
        n_cols = len(con.execute(
            f"DESCRIBE SELECT * FROM read_csv('{out.resolve()}', header=true, "
            "delim=';', quote='\"', escape='\"', all_varchar=true, "
            "encoding='utf-8', strict_mode=false)"
        ).fetchall())
    finally:
        con.close()
    for p in paths:
        p.unlink()
    if n_out != linhas:
        raise SystemExit(f"[ERRO 2020] entrada={linhas} != saida={n_out}")
    return {"arquivo": out, "registros": n_out,
            "meses": [f"{m:02d}{s}" for (m, s), _, _ in itens],
            "cols": n_cols, "delim": ";"}


# ---------- orquestração ----------

def processar_ano(ano: str) -> dict:
    itens = coletar_meses(ano)
    if not itens:
        raise SystemExit(f"[ERRO] nenhum mês encontrado para {ano} em google_drive/*.zip")

    presentes = sorted({m for (m, _), _, _ in itens})
    inexistentes = [m for m in MESES_INEXISTENTES.get(int(ano), []) if m not in presentes]
    esperados = [m for m in range(1, 13) if m not in inexistentes]
    faltando = [m for m in esperados if m not in presentes]
    if faltando:
        print(f"  [AVISO {ano}] MESES AUSENTES: {faltando} "
              f"(presentes: {[f'{m:02d}{s}' for (m, s), _, _ in itens]})")
    if inexistentes:
        print(f"  [INFO {ano}] mês(es) inexistente(s) na base (ignorado): "
              f"{inexistentes}")

    tmp_dir = TMP / ano
    tmp_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    try:
        if ano == "2020":
            res = unificar_2020(itens, tmp_dir)
        else:
            res = unificar_uniforme(ano, itens, DELIM.get(int(ano), DELIM_PADRAO), tmp_dir)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    dt = time.time() - t0
    mb = res["arquivo"].stat().st_size / 1e6
    print(f"OK {ano}: {res['registros']:,} registros | {len(res['meses'])} meses "
          f"| {res['cols']} colunas | delim='{res['delim']}' | "
          f"{res['arquivo'].name} ({mb:.1f} MB, {dt:.1f}s)")
    if faltando:
        print(f"   !! {ano} INCOMPLETO — falta(m) mês(es) {faltando} — "
              f"reenviar para o Drive antes do tratamento")

    # limpeza: só após validação
    zips_ano = [z for z in DRIVE.glob("*.zip") if re.search(ano, z.name)]
    for z in zips_ano:
        z.unlink()
    res["faltando"] = faltando
    res["zips_removidos"] = len(zips_ano)
    return res


def main(anos: list[str]):
    TMP.mkdir(parents=True, exist_ok=True)
    resumos = []
    for ano in anos:
        print(f"--- {ano} ---")
        resumos.append((ano, processar_ano(ano)))
    print("\n=== RESUMO ===")
    total = 0
    for ano, r in resumos:
        flag = "  <<<< INCOMPLETO" if r["faltando"] else ""
        print(f"  {ano}: {r['registros']:>12,} registros | {r['cols']} cols | "
              f"{r['arquivo'].name}{flag}")
        total += r["registros"]
    print(f"  TOTAL: {total:,} registros")


if __name__ == "__main__":
    if len(sys.argv) > 1:
        anos = sys.argv[1:]
    else:
        anos = [str(y) for y in range(2018, 2026)]
    main(anos)

Overwriting scripts/02_extrair_unificar.py


### 2.5 Normalização → schema canônico de 25 colunas (`03`)

Aplica em 19/19 anos:
- **mapeamento de nomes** (NFKD-fold) das três eras → 25 colunas na ordem fixa;
- `SELECT` canônico com `all_varchar` → TRIM, tokens (`NA/N-I/N-A`/vazio) → `NULL`,
  datas → `AAAA-MM-DD` (defensivo `DD/MM`), `hora` → `HH:MM:SS`;
- validação: contagem entrada == saída, header md5 único (`8fe1a7bb…`);
- **fallback automático `parallel=false`** quando o reader paralelo do DuckDB recusa o
  arquivo (caso `infracoes_2023.csv.gz`) — leitura serial dentro do limite de memória;
- grava `.csv` (depois `gzip -6` → `.csv.gz`) com `os.replace` atômico.

**Total fase 1: 92.215.588 linhas · 19 anos · header idêntico.**

In [6]:
%%writefile scripts/03_normalizar.py
#!/usr/bin/env python3
"""
Etapa 03 — Normalização dos CSVs unificados de `dados_brutos`.

Padroniza TODOS os anos (2007–2025) para um schema canônico único:

  * 25 colunas canônicas (união dos schemas das três eras), ordem fixa,
    nomes em snake_case
  * Delimitador único `;`, UTF-8, LF, sem BOM, aspas mínimas
  * Datas em AAAA-MM-DD (converte DD/MM/AAAA se aparecer; já é o padrão)
  * `hora` em HH:MM:SS (anos que só tinham o dígito da hora → 'HH:00:00')
  * TRIM em todos os campos
  * Tokens de ausência ('NA', 'N/I', 'N/A' e vazio) → vazio (NULL)

Validação por ano (obrigatória): contagem de entrada == contagem de saída,
`data_infracao` 100% ISO e `hora` 100% HH:MM:SS no arquivo normalizado.
Só depois disso o temporário substitui o arquivo final.

Aceita fonte `.csv` ou `.csv.gz` (a etapa de compactação prévia libera disco).
Saída: sobrescreve `dados_brutos/infracoes_<ANO>.csv` (via `.norm.csv` + replace).

Uso:
  .venv/bin/python scripts/03_normalizar.py              # todos os anos
  .venv/bin/python scripts/03_normalizar.py 2007 2008    # só os indicados
"""
from __future__ import annotations

import csv
import gzip
import os
import shutil
import sys
import time
import unicodedata
from pathlib import Path

import duckdb

BASE = Path(__file__).resolve().parent.parent
BRUTOS = BASE / "dados_brutos"

# ---------------------------------------------------------------------------
# Schema canônico (união de todas as eras, ordem fixa)
# ---------------------------------------------------------------------------
CANONICA = [
    "numero_auto",           # era2/3 (2019+)            — era1: ausente
    "data_infracao",         # era1: dat_infracao
    "hora",                  # era1 2015+                — 2007–2014: ausente
    "tip_abordagem",
    "ind_assinou_auto",
    "ind_veiculo_estrangeiro",
    "sentido_trafego",       # era1: ind_sentido_trafego
    "uf_placa",              # 2021: ausente
    "uf_infracao",
    "num_br_infracao",
    "num_km_infracao",
    "nom_municipio",
    "cod_infracao",          # era1 2017+                — 2007–2016: ausente
    "descricao_abreviada",
    "enquadramento",
    "data_inicio_vigencia",
    "data_fim_vigencia",
    "med_realizada",         # era2/3: Medição Infração
    "med_considerada",
    "exc_verificado",
    "especie",
    "nome_veiculo_marca",
    "nom_modelo_veiculo",    # era2 2019–2021: ausente
    "tipo_veiculo",          # só era3 (2022+)
    "qtd_infracoes",         # era2 2020+ / era3
]

DATAS = {"data_infracao", "data_inicio_vigencia", "data_fim_vigencia"}
TOKENS_AUSENCIA = ("", "NA", "N/I", "N/A")


def _n(nome: str) -> str:
    """Normaliza nome de coluna: sem acento, minúsculo, sem espaços extras."""
    s = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return " ".join(s.lower().split())


# Mapeia nome-de-coluna-normalizado -> coluna canônica (inclui identidade p/ re-execução)
MAPEAMENTO: dict[str, str] = {c: c for c in CANONICA}
MAPEAMENTO.update({
    # era 1 (2007–2018, snake_case)
    "dat_infracao": "data_infracao",
    "ind_sentido_trafego": "sentido_trafego",
    # era 2/3 (2019–2025, pt-BR)
    "numero do auto": "numero_auto",
    "data da infracao (dd/mm/aaaa)": "data_infracao",
    "indicador de abordagem": "tip_abordagem",
    "assinatura do auto": "ind_assinou_auto",
    "indicador veiculo estrangeiro": "ind_veiculo_estrangeiro",
    "sentido trafego": "sentido_trafego",
    "uf placa": "uf_placa",
    "uf infracao": "uf_infracao",
    "br infracao": "num_br_infracao",
    "km infracao": "num_km_infracao",
    "municipio": "nom_municipio",
    "codigo da infracao": "cod_infracao",
    "descricao abreviada infracao": "descricao_abreviada",
    "enquadramento da infracao": "enquadramento",
    "inicio vigencia da infracao": "data_inicio_vigencia",
    "fim vigencia infracao": "data_fim_vigencia",
    "medicao infracao": "med_realizada",
    "medicao considerada": "med_considerada",
    "excesso verificado": "exc_verificado",
    "descricao especie veiculo": "especie",
    "descricao marca veiculo": "nome_veiculo_marca",
    "descricao modelo veiculo": "nom_modelo_veiculo",
    "descricao tipo veiculo": "tipo_veiculo",
    "hora infracao": "hora",
    "qtd infracoes": "qtd_infracoes",
})


def _q(ident: str) -> str:
    return '"' + ident.replace('"', '""') + '"'


def _expr_data(q: str) -> str:
    in_list = ", ".join(f"'{t}'" for t in TOKENS_AUSENCIA)
    return (
        f"CASE WHEN trim({q}) IN ({in_list}) THEN NULL "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$') THEN trim({q}) "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{2}}/[0-9]{{2}}/[0-9]{{4}}$') "
        f"THEN substr(trim({q}), 7, 4) || '-' || substr(trim({q}), 4, 2) || '-' || substr(trim({q}), 1, 2) "
        f"ELSE trim({q}) END"
    )


def _expr_hora(q: str) -> str:
    in_list = ", ".join(f"'{t}'" for t in TOKENS_AUSENCIA)
    return (
        f"CASE WHEN trim({q}) IN ({in_list}) THEN NULL "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{2}}:[0-9]{{2}}:[0-9]{{2}}$') THEN trim({q}) "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{1,2}}$') "
        f"AND try_cast(trim({q}) AS INTEGER) BETWEEN 0 AND 23 "
        f"THEN lpad(trim({q}), 2, '0') || ':00:00' "
        f"ELSE trim({q}) END"
    )


def _expr_comum(q: str) -> str:
    in_list = ", ".join(f"'{t}'" for t in TOKENS_AUSENCIA)
    return f"CASE WHEN trim({q}) IN ({in_list}) THEN NULL ELSE trim({q}) END"


def _fonte(ano: int) -> Path:
    gz = BRUTOS / f"infracoes_{ano}.csv.gz"
    csv_p = BRUTOS / f"infracoes_{ano}.csv"
    if gz.exists():
        return gz
    if csv_p.exists():
        return csv_p
    raise SystemExit(f"[ERRO] fonte não encontrada para {ano} ({csv_p.name} / {gz.name})")


def _ler_cabecalho(src: Path) -> tuple[list[str], str]:
    opener = gzip.open if src.suffix == ".gz" else open
    with opener(src, "rb") as fh:
        cab = fh.readline().decode("utf-8", "replace").rstrip("\r\n")
    # delim detectado pelo próprio cabeçalho (funciona também p/ re-execução)
    delim = ";" if ";" in cab else ","
    cols = next(csv.reader([cab], delimiter=delim))
    return cols, delim


def normalizar_ano(ano: int) -> tuple[int, float]:
    src = _fonte(ano)
    dst = BRUTOS / f"infracoes_{ano}.csv"
    tmp = dst.with_name(f"infracoes_{ano}.norm.csv")
    if tmp.exists():
        tmp.unlink()

    cols, delim = _ler_cabecalho(src)

    # ---- monta o SELECT no schema canônico -------------------------------
    origem: dict[str, str] = {}
    desconhecidas: list[str] = []
    for c in cols:
        chave = _n(c)
        if chave not in MAPEAMENTO:
            desconhecidas.append(c)
            continue
        alvo = MAPEAMENTO[chave]
        if alvo in origem:
            raise SystemExit(
                f"[ERRO] {ano}: duas colunas de origem mapeiam para '{alvo}': "
                f"'{origem[alvo]}' e '{c}'"
            )
        origem[alvo] = c
    if desconhecidas:
        raise SystemExit(f"[ERRO] {ano}: colunas de origem sem mapeamento: {desconhecidas}")

    partes: list[str] = []
    for alvo in CANONICA:
        if alvo not in origem:
            partes.append(f'CAST(NULL AS VARCHAR) AS "{alvo}"')
            continue
        q = _q(origem[alvo])
        if alvo in DATAS:
            e = _expr_data(q)
        elif alvo == "hora":
            e = _expr_hora(q)
        else:
            e = _expr_comum(q)
        partes.append(f'{e} AS "{alvo}"')
    selecao = ", ".join(partes)

    con = duckdb.connect()
    # tuning p/ máquina pequena (2 cores / 3,8 GB): 1 thread + sem buffer de
    # ordenação (com 1 thread a ordem de origem é preservada) + spill em disco
    (BRUTOS / "_ddb_tmp").mkdir(exist_ok=True)
    con.execute("SET threads=1")
    con.execute("SET preserve_insertion_order=false")
    con.execute("SET memory_limit='2400MB'")
    con.execute(f"SET temp_directory='{BRUTOS / '_ddb_tmp'}'")
    caminho = str(src.resolve())
    leitura = (
        f"read_csv('{caminho}', header=true, delim='{delim}', quote='\"', escape='\"', "
        "all_varchar=true, encoding='utf-8', strict_mode=false, parallel=true)"
    )

    t0 = time.time()
    try:
        n_in = con.execute(f"SELECT count(*) FROM {leitura}").fetchone()[0]
    except duckdb.NotImplementedException:
        # alguns arquivos nao suportam o CSV reader paralelo ("full read");
        # cai para leitura serial (bufferiza o descompactado — só viável com
        # arquivos < limite de memória do DuckDB)
        leitura = (
            f"read_csv('{caminho}', header=true, delim='{delim}', quote='\"', escape='\"', "
            "all_varchar=true, encoding='utf-8', strict_mode=false, parallel=false)"
        )
        print(f"  [INFO {ano}] reader paralelo indisponível — usando leitura serial")
        n_in = con.execute(f"SELECT count(*) FROM {leitura}").fetchone()[0]

    con.execute(
        f"COPY (SELECT {selecao} FROM {leitura}) TO '{tmp.resolve()}' "
        "(HEADER true, DELIMITER ';', QUOTE '\"', ESCAPE '\"', NULL '')"
    )

    leitura_tmp = (
        f"read_csv('{tmp.resolve()}', header=true, delim=';', quote='\"', escape='\"', "
        "all_varchar=true, encoding='utf-8', strict_mode=false, parallel=true)"
    )
    n_out, bad_data, bad_hora = con.execute(
        f"SELECT count(*), "
        f"coalesce(sum(CASE WHEN data_infracao IS NOT NULL AND data_infracao <> '' "
        f"AND NOT regexp_matches(data_infracao, '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$') "
        f"THEN 1 ELSE 0 END), 0), "
        f"coalesce(sum(CASE WHEN hora IS NOT NULL AND hora <> '' "
        f"AND NOT regexp_matches(hora, '^[0-9]{{2}}:[0-9]{{2}}:[0-9]{{2}}$') "
        f"THEN 1 ELSE 0 END), 0) "
        f"FROM {leitura_tmp}"
    ).fetchone()
    con.close()

    if n_in != n_out:
        tmp.unlink(missing_ok=True)
        raise SystemExit(f"[ERRO] {ano}: contagem divergente — entrada {n_in:,} vs saída {n_out:,}")
    if bad_data or bad_hora:
        tmp.unlink(missing_ok=True)
        raise SystemExit(
            f"[ERRO] {ano}: valores fora do padrão no normalizado — "
            f"data_infracao={bad_data}, hora={bad_hora}"
        )

    # ---- troca definitiva -----------------------------------------------
    os.replace(tmp, dst)
    if src != dst:            # remove o .gz depois da troca validada
        src.unlink()

    dt = time.time() - t0
    mb = dst.stat().st_size / 1e6
    ausentes = [c for c in CANONICA if c not in origem]
    info = f" | ausentes(vazio): {', '.join(ausentes)}" if ausentes else ""
    print(
        f"OK {ano}: {n_out:,} registros | 25 colunas | delim=';' | "
        f"{dst.name} ({mb:,.1f} MB, {dt:.1f}s){info}",
        flush=True,
    )
    livre = shutil.disk_usage(BRUTOS).free / 1e9
    print(f"     disco livre: {livre:.2f} GB", flush=True)
    return n_out, dt


def main(argv: list[str]) -> int:
    if argv:
        anos = [int(a) for a in argv]
    else:
        anos = sorted(
            int(p.stem.split("_")[1].split(".")[0])
            for p in BRUTOS.glob("infracoes_*.csv*")
            if not p.name.endswith(".norm.csv")
        )
    total = 0
    t0 = time.time()
    for ano in anos:
        n, _ = normalizar_ano(ano)
        total += n
    print(f"\n=== RESUMO NORMALIZAÇÃO ===")
    print(f"  anos: {min(anos)}–{max(anos)} ({len(anos)} arquivos)")
    print(f"  registros: {total:,}")
    print(f"  tempo total: {time.time() - t0:.1f}s")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))

Overwriting scripts/03_normalizar.py


## 3. Fase 2 — tratamento e implementação de colunas (`04_tratar.py`)

Le os 19 `.csv.gz` (1 passada de gzip/ano em tabela TEMP `stage`; fallback serial no 2023)
e grava **`dados_tratados/base_analitica/infracoes_YYYY.parquet`** (ZSTD, **1,4 GB**).

### 3.1 Padronização (25 colunas de origem → todas MAIÚSCULAS)

| Regra | Detalhe |
|-------|---------|
| Sentinela → `NULL` | `'-1`, `-1`, `NA`, `N/A`, `N/I`, `N-I`, `N-A`, `NULL`, `NULO`, `#N/A` |
| Texto | `TRIM` + **MAIÚSCULAS** + acentos→ASCII (`TRAÇÃO`=`TRACAO`; `SÃO PAULO`=`SAO PAULO`) |
| `DATA_*` | `DATE` (ISO ou `DD/MM/AAAA` defensivo; inválida → `NULL`) |
| `HORA` | `TIME` |
| `NUM_KM_INFRACAO` | `DOUBLE` (aceita vírgula; negativo → `NULL`) |
| `QTD_INFRACOES` / `NUM_BR_INFRACAO` | `BIGINT` (`<=0` / não numérico → `NULL`) |
| `UF_PLACA`/`UF_INFRACAO` | só 27 UFs válidas → resto `NULL` (3.941.160 células) |
| `COD_INFRACAO` | mantido como **código texto** |

### 3.2 Colunas derivadas (25 → **28**)

- **`MES`** (`VARCHAR`): **nome do mês** — `JANEIRO…DEZEMBRO` (acento→ASCII: `MARCO`).
  *(alteração: era SMALLINT 1–12; trocado a pedido)*
- **`ANO`** (`SMALLINT`): ano de `DATA_INFRACAO`.
- **`QUARTIL`** (`VARCHAR`): **4 períodos de 3 meses** — `Q1`=jan-mar, `Q2`=abr-jun,
  `Q3`=jul-set, `Q4`=out-dez *(pedido: "a cada 4 meses" com Q1–Q4 = 4×3 meses)*.

**Nenhuma linha removida: 92.215.588 entram e saem.**

In [7]:
%%writefile scripts/04_tratar.py
#!/usr/bin/env python3
"""
04_tratar.py — Fase 2: tratamento e padronizacao -> BASE ANALITICA (Parquet).

Le os 19 CSV normalizados (dados_brutos/infracoes_*.csv.gz) e gera:
  dados_tratados/base_analitica/infracoes_YYYY.parquet   (28 colunas, ZSTD)
  dados_tratados/resumo_colunas.md                       (estatisticas + resumo)

Padronizacao (todas as colunas de texto):
  sentinela ('-1, -1, NA, N/A, N/I, N-I, N-A, NULL, NULO...) -> NULL
  senao: TRIM -> UPPER -> acentos -> ASCII -> '' -> NULL
Tipos:
  DATA_*        DATE  (ISO 'AAAA-MM-DD' ou 'DD/MM/AAAA' defensivo; invalido -> NULL)
  HORA          TIME  (invalido -> NULL)
  NUM_KM        DOUBLE (virgula ou ponto aceito; negativo -> NULL)
  QTD_INFRACOES BIGINT (<= 0 -> NULL)
  NUM_BR        BIGINT (nao numerico -> NULL)
  COD_INFRACAO  VARCHAR (e' codigo, nao quantidade)
  UF_*          VARCHAR (apenas 27 UFs validas; resto -> NULL)
Derivadas (a partir de DATA_INFRACAO):
  MES     VARCHAR nome do mes (JANEIRO..DEZEMBRO; acento->ASCII: MARCO)
  ANO     SMALLINT (ano da infracao)
  QUARTIL Q1=jan-mar | Q2=abr-jun | Q3=jul-set | Q4=out-dez
Linhas: NENHUMA removida (92.215.588 entram e saem).

Uso: .venv/bin/python scripts/04_tratar.py [ANO ...] [--resumo]
"""
from __future__ import annotations

import json
import sys
import time
import unicodedata
from pathlib import Path

import duckdb

RAIZ = Path(__file__).resolve().parent.parent
BRUTOS = RAIZ / "dados_brutos"
TRATADOS = RAIZ / "dados_tratados"
BASE_DIR = TRATADOS / "base_analitica"
TMP_DIR = TRATADOS / "_ddb_tmp"
VALIDACAO = TRATADOS / "_validacao.jsonl"

TOTAL_ESPERADO = 92_215_588

# ---------------------------------------------------------------------------
# mapeamento de acentos (construido com unicodedata para garantir paridade)
# ---------------------------------------------------------------------------
_par: dict[str, str] = {}
for _ch in "ÁÀÂÃÄÅÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑáàâãäåéèêëíìîïóòôõöúùûüçñ":
    _base = unicodedata.normalize("NFD", _ch)[0]
    _par[_ch] = _base
ACC_SRC = "".join(_par.keys())
ACC_DST = "".join(_par.values())

SENTINELAS = ("-1", "'-1", "NA", "N/A", "N/I", "N-I", "N-A", "NULL", "NULO", "#N/A")
SENT_SQL = ", ".join("'" + s.replace("'", "''") + "'" for s in SENTINELAS)

UFS = (
    "AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO"
).split()
UFS_SQL = ", ".join("'" + u + "'" for u in UFS)

UFS_NAO_NUM = f"({UFS_SQL})"


def p(col: str) -> str:
    """Padroniza texto: sentinela -> NULL; senao TRIM+UPPER+dobrar acentos; '' -> NULL."""
    v = f"upper(trim({col}))"
    return (
        f"CASE WHEN {col} IS NULL OR {v} IN ({SENT_SQL}) THEN NULL "
        f"ELSE NULLIF(translate({v}, '{ACC_SRC}', '{ACC_DST}'), '') END"
    )


def ufs(col: str) -> str:
    """UF: apenas siglas validas (27 UFs); sentinela/lixo -> NULL."""
    v = f"upper(trim({col}))"
    return f"CASE WHEN {v} IN {UFS_NAO_NUM} THEN {v} END"


def dt(col: str) -> str:
    """Data: ISO ou DD/MM/AAAA defensivo; invalida -> NULL."""
    return (
        f"CASE WHEN {col} IS NULL THEN NULL "
        f"WHEN trim({col}) ~ '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$' "
        f"THEN try_cast(trim({col}) AS DATE) "
        f"WHEN trim({col}) ~ '^[0-9]{{2}}/[0-9]{{2}}/[0-9]{{4}}$' "
        f"THEN try_strptime(trim({col}), '%d/%m/%Y')::DATE "
        f"ELSE NULL END"
    )


HORA = "try_cast(trim(hora) AS TIME)"
KM = "try_cast(nullif(replace(trim(num_km_infracao), ',', '.'), '') AS DOUBLE)"
KM_OK = f"CASE WHEN {KM} IS NULL OR {KM} < 0 THEN NULL ELSE {KM} END"
QTD = "try_cast(trim(qtd_infracoes) AS BIGINT)"
QTD_OK = f"CASE WHEN {QTD} IS NULL OR {QTD} <= 0 THEN NULL ELSE {QTD} END"
BR = "try_cast(trim(num_br_infracao) AS BIGINT)"

COLS_SELECT = f"""
    {p('numero_auto')}                          AS NUMERO_AUTO,
    {dt('data_infracao')}                       AS DATA_INFRACAO,
    {HORA}                                      AS HORA,
    {p('tip_abordagem')}                        AS TIP_ABORDAGEM,
    {p('ind_assinou_auto')}                     AS IND_ASSINOU_AUTO,
    {p('ind_veiculo_estrangeiro')}              AS IND_VEICULO_ESTRANGEIRO,
    {p('sentido_trafego')}                      AS SENTIDO_TRAFEGO,
    {ufs('uf_placa')}                           AS UF_PLACA,
    {ufs('uf_infracao')}                        AS UF_INFRACAO,
    {BR}                                        AS NUM_BR_INFRACAO,
    {KM_OK}                                     AS NUM_KM_INFRACAO,
    {p('nom_municipio')}                        AS NOM_MUNICIPIO,
    {p('cod_infracao')}                         AS COD_INFRACAO,
    {p('descricao_abreviada')}                  AS DESCRICAO_ABRVIADA,
    {p('enquadramento')}                        AS ENQUADRAMENTO,
    {dt('data_inicio_vigencia')}                AS DATA_INICIO_VIGENCIA,
    {dt('data_fim_vigencia')}                   AS DATA_FIM_VIGENCIA,
    {p('med_realizada')}                        AS MED_REALIZADA,
    {p('med_considerada')}                      AS MED_CONSIDERADA,
    {p('exc_verificado')}                       AS EXC_VERIFICADO,
    {p('especie')}                              AS ESPECIE,
    {p('nome_veiculo_marca')}                   AS NOME_VEICULO_MARCA,
    {p('nom_modelo_veiculo')}                   AS NOM_MODELO_VEICULO,
    {p('tipo_veiculo')}                         AS TIPO_VEICULO,
    {QTD_OK}                                    AS QTD_INFRACOES
"""

SQL_TRATAR = f"""
SELECT
    {COLS_SELECT},
    CASE MES_NUM
        WHEN 1  THEN 'JANEIRO'
        WHEN 2  THEN 'FEVEREIRO'
        WHEN 3  THEN 'MARCO'
        WHEN 4  THEN 'ABRIL'
        WHEN 5  THEN 'MAIO'
        WHEN 6  THEN 'JUNHO'
        WHEN 7  THEN 'JULHO'
        WHEN 8  THEN 'AGOSTO'
        WHEN 9  THEN 'SETEMBRO'
        WHEN 10 THEN 'OUTUBRO'
        WHEN 11 THEN 'NOVEMBRO'
        WHEN 12 THEN 'DEZEMBRO'
    END AS MES,
    ANO,
    CASE
        WHEN MES_NUM IS NULL THEN NULL
        WHEN MES_NUM <= 3  THEN 'Q1'
        WHEN MES_NUM <= 6  THEN 'Q2'
        WHEN MES_NUM <= 9  THEN 'Q3'
        ELSE 'Q4'
    END AS QUARTIL
FROM (
    SELECT
        *,
        EXTRACT(MONTH FROM {dt('data_infracao')})::SMALLINT AS MES_NUM,
        EXTRACT(YEAR  FROM {dt('data_infracao')})::SMALLINT AS ANO
    FROM stage
)
"""

SQL_VALIDACAO = f"""
SELECT
    count(*)                                                          AS linhas,
    count(*) FILTER (WHERE data_infracao IS NOT NULL AND _d  IS NULL) AS data_ruim,
    count(*) FILTER (WHERE data_infracao IS NOT NULL AND _d  IS NOT NULL
                      AND EXTRACT(YEAR FROM _d)::INT <> {{ANO}})       AS ano_divergente,
    count(*) FILTER (WHERE hora IS NOT NULL AND _h IS NULL)           AS hora_ruima,
    count(*) FILTER (WHERE uf_placa IS NOT NULL AND _ufp IS NULL)     AS uf_placa_invalida,
    count(*) FILTER (WHERE uf_infracao IS NOT NULL AND _ufi IS NULL)  AS uf_infracao_invalida,
    count(*) FILTER (WHERE num_km_infracao IS NOT NULL AND _km IS NULL) AS km_invalido,
    count(*) FILTER (WHERE qtd_infracoes IS NOT NULL AND _q IS NULL)  AS qtd_invalida,
    count(*) FILTER (WHERE num_br_infracao IS NOT NULL AND _br IS NULL) AS br_invalido,
    count(*) FILTER (WHERE upper(trim(especie)) IN ({SENT_SQL})) AS especie_sentinela
FROM (
    SELECT
        data_infracao, hora, uf_placa, uf_infracao,
        num_km_infracao, qtd_infracoes, num_br_infracao, especie,
        {dt('data_infracao')} AS _d,
        {HORA}                AS _h,
        {ufs('uf_placa')}     AS _ufp,
        {ufs('uf_infracao')}  AS _ufi,
        {KM_OK}               AS _km,
        {QTD_OK}              AS _q,
        {BR}                  AS _br
    FROM stage
)
"""


def criar_stage(con: duckdb.DuckDBPyConnection, arquivo: Path, paralelo: bool = True) -> None:
    """Materializa o CSV (1x decompress) em TEMP; metricas e COPY leem daqui."""
    q = chr(34)  # aspas duplas dentro do SQL: quote='"'
    opts = (
        "header=true, delim=';', all_varchar=true, strict_mode=false, "
        + "quote='" + q + "', escape='" + q + "', "
        + "parallel=" + ("true" if paralelo else "false")
    )
    con.execute("DROP TABLE IF EXISTS stage")
    con.execute(
        f"CREATE TEMP TABLE stage AS "
        f"SELECT * FROM read_csv('{arquivo}', {opts})"
    )


def tratar_ano(con: duckdb.DuckDBPyConnection, ano: int) -> dict:
    arquivo = BRUTOS / f"infracoes_{ano}.csv.gz"
    if not arquivo.exists():
        raise FileNotFoundError(
            f"{arquivo} nao encontrado (remonte as partes: cat {arquivo.name}.* > {arquivo.name})"
        )
    destino = BASE_DIR / f"infracoes_{ano}.parquet"
    destino.unlink(missing_ok=True)

    try:
        criar_stage(con, arquivo, paralelo=True)
        metricas = con.execute(SQL_VALIDACAO.replace('{ANO}', str(ano))).fetchone()
        con.execute(
            f"COPY ({SQL_TRATAR}) TO '{destino}' "
            f"(FORMAT PARQUET, COMPRESSION 'ZSTD')"
        )
    except duckdb.NotImplementedException:
        print(f"  [{ano}] reader paralelo recusou o arquivo; retry serial", flush=True)
        criar_stage(con, arquivo, paralelo=False)
        destino.unlink(missing_ok=True)
        metricas = con.execute(SQL_VALIDACAO.replace('{ANO}', str(ano))).fetchone()
        con.execute(
            f"COPY ({SQL_TRATAR}) TO '{destino}' "
            f"(FORMAT PARQUET, COMPRESSION 'ZSTD')"
        )

    con.execute("DROP TABLE IF EXISTS stage")
    linhas = con.execute(f"SELECT count(*) FROM read_parquet('{destino}')").fetchone()[0]
    if linhas != metricas[0]:
        raise AssertionError(f"[{ano}] contagem divergente: fonte={metricas[0]} parquet={linhas}")

    nomes = [
        "linhas", "data_ruim", "ano_divergente", "hora_ruima",
        "uf_placa_invalida", "uf_infracao_invalida", "km_invalido",
        "qtd_invalida", "br_invalido", "especie_sentinela",
    ]
    m = dict(zip(nomes, metricas))
    m["ano"] = ano
    m["parquet_mb"] = round(destino.stat().st_size / 1e6, 1)
    return m


# ---------------------------------------------------------------------------
# fase B — resumo das colunas
# ---------------------------------------------------------------------------
def fase_b(con: duckdb.DuckDBPyConnection, validacoes: list[dict]) -> Path:
    glob = str(BASE_DIR / "*.parquet")
    con.execute(f"CREATE OR REPLACE VIEW base AS SELECT * FROM read_parquet('{glob}')")
    total = con.execute("SELECT count(*) FROM base").fetchone()[0]
    if total != TOTAL_ESPERADO:
        raise AssertionError(f"TOTAL global {total:,} != esperado {TOTAL_ESPERADO:,}")

    cols = [
        r[0] for r in con.execute(
            "SELECT column_name FROM information_schema.columns "
            "WHERE table_name='base' ORDER BY ordinal_position"
        ).fetchall()
    ]
    tipos = dict(con.execute(
        "SELECT column_name, data_type FROM information_schema.columns "
        "WHERE table_name='base'"
    ).fetchall())

    aggs = ["count(*) AS n"]
    for c in cols:
        aggs.append(f"count(*) FILTER (WHERE \"{c}\" IS NULL) AS \"nul_{c}\"")
        aggs.append(f"approx_count_distinct(\"{c}\") AS \"d_{c}\"")
    row = dict(zip(
        ["n"] + [x for c in cols for x in (f"nul_{c}", f"d_{c}")],
        con.execute(f"SELECT {', '.join(aggs)} FROM base").fetchone(),
    ))

    # nulos por era (era1 2007-18 | era2 2019-21 | era3 2022-25)
    eras = {"era1": "ANO BETWEEN 2007 AND 2018", "era2": "ANO BETWEEN 2019 AND 2021",
            "era3": "ANO >= 2022"}
    aggs_e = []
    for c in cols:
        for e, w in eras.items():
            aggs_e.append(
                f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL AND {w}) "
                f"/ nullif(count(*) FILTER (WHERE {w}), 0), 1) AS \"{e}__{c}\""
            )
    row_e = dict(zip(
        [f"{e}__{c}" for c in cols for e in eras],
        con.execute(f"SELECT {', '.join(aggs_e)} FROM base").fetchone(),
    ))

    # redundancias / resumos possiveis
    red = con.execute("""
        SELECT
            approx_count_distinct(COD_INFRACAO) AS n_cod,
            approx_count_distinct(COD_INFRACAO || '|' || coalesce(ENQUADRAMENTO,'')) AS n_cod_enq,
            approx_count_distinct(COD_INFRACAO || '|' || coalesce(DESCRICAO_ABRVIADA,'')) AS n_cod_desc,
            approx_count_distinct(ENQUADRAMENTO || '|' || coalesce(DESCRICAO_ABRVIADA,'')) AS n_enq_desc,
            approx_count_distinct(NOME_VEICULO_MARCA) AS n_marca,
            approx_count_distinct(NOM_MODELO_VEICULO) AS n_modelo,
            approx_count_distinct(NOM_MUNICIPIO) AS n_municipio,
            count(*) FILTER (WHERE MED_REALIZADA IS NOT NULL
                 AND try_cast(replace(MED_REALIZADA,',','.') AS DOUBLE) IS NULL) AS med_ruim,
            count(*) FILTER (WHERE MED_REALIZADA IS NOT NULL) AS med_nao_nulo,
            count(*) FILTER (WHERE MED_CONSIDERADA IS NOT NULL
                 AND try_cast(replace(MED_CONSIDERADA,',','.') AS DOUBLE) IS NULL) AS medc_ruim,
            count(*) FILTER (WHERE MED_CONSIDERADA IS NOT NULL) AS medc_nao_nulo
        FROM base
    """).fetchone()
    red = dict(zip(
        ["n_cod", "n_cod_enq", "n_cod_desc", "n_enq_desc", "n_marca", "n_modelo",
         "n_municipio", "med_ruim", "med_nao_nulo", "medc_ruim", "medc_nao_nulo"], red))

    # amostras de MED_*
    amostras = con.execute("""
        SELECT MED_REALIZADA, count(*) FROM base
        WHERE MED_REALIZADA IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 6
    """).fetchall()

    # ------------------------------------------------------------------ md
    def pct(c: str) -> str:
        return f"{100.0 * row[f'nul_{c}'] / row['n']:.2f}%"

    linhas_md = [
        "# resumo_colunas — base analítica (dados_tratados)",
        "",
        f"- **Linhas:** {row['n']:,} · **Colunas:** {len(cols)} · "
        f"Gerado por `scripts/04_tratar.py` em {time.strftime('%Y-%m-%d %H:%M')}.",
        "- **Fonte:** `dados_brutos/infracoes_2007..2025.csv.gz` (19 arquivos, schema canônico 25 col).",
        "- Colunas de texto: sentinela → `NULL`; `TRIM` + `MAIÚSCULAS` + acentos→ASCII. "
        "Colunas derivadas: `MES` (nome do mês: JANEIRO…DEZEMBRO), `ANO`, `QUARTIL` (Q1=jan-mar, Q2=abr-jun, Q3=jul-set, Q4=out-dez).",
        "",
        "## 1. Estatísticas globais",
        "",
        "| # | Coluna | Tipo | Nulos | Nulos % | Distintos (aprox) |",
        "|---|--------|------|------:|--------:|------------------:|",
    ]
    for i, c in enumerate(cols, 1):
        linhas_md.append(
            f"| {i} | `{c}` | {tipos[c]} | {row[f'nul_{c}']:,} | {pct(c)} | {row[f'd_{c}']:,} |"
        )

    linhas_md += [
        "",
        "## 2. Nulos por era (era1 2007–2018 · era2 2019–2021 · era3 2022–2025)",
        "",
        "| Coluna | era1 | era2 | era3 |",
        "|--------|-----:|-----:|-----:|",
    ]
    for c in cols:
        v = [row_e[f"{e}__{c}"] for e in eras]
        linhas_md.append(f"| `{c}` | {v[0]}% | {v[1]}% | {v[2]}% |")

    linhas_md += ["", "## 3. Anomalias corrigidas na padronização", "",
                  "| Ano | Linhas | Datas inválidas | Ano divergente | Hora inválida | UF placa inv. | UF infra inv. | KM inv. | Qtd inv. | BR inv. | Espécie sentinela | Parquet MB |",
                  "|----:|-------:|----------------:|---------------:|--------------:|--------------:|--------------:|--------:|---------:|--------:|------------------:|-----------:|"]
    for m in validacoes:
        linhas_md.append(
            f"| {m['ano']} | {m['linhas']:,} | {m['data_ruim']} | {m['ano_divergente']} | "
            f"{m['hora_ruima']} | {m['uf_placa_invalida']} | {m['uf_infracao_invalida']} | "
            f"{m['km_invalido']} | {m['qtd_invalida']} | {m['br_invalido']} | "
            f"{m['especie_sentinela']} | {m['parquet_mb']} |"
        )

    # ---------------------------------------------------------- recomendacoes
    fd_enq = red["n_cod_enq"] <= red["n_cod"] * 1.02
    fd_desc = red["n_cod_desc"] <= red["n_cod"] * 1.02
    med_ok = red["med_nao_nulo"] > 0 and red["med_ruim"] == 0
    medc_ok = red["medc_nao_nulo"] > 0 and red["medc_ruim"] == 0

    linhas_md += [
        "",
        "## 4. Colunas que PODEM ser resumidas",
        "",
        f"- **`DESCRICAO_ABRVIADA` / `ENQUADRAMENTO` vs `COD_INFRACAO`**: "
        f"{red['n_cod']:,} códigos distintos; pares (código,enquadramento) ≈ {red['n_cod_enq']:,}; "
        f"pares (código,descrição) ≈ {red['n_cod_desc']:,}; "
        f"(enquadramento,descrição) ≈ {red['n_enq_desc']:,}. "
        + ("Os textos são **funcionalmente dependentes do código** → em bases "
           "modeláveis podem ser **resumidos em `COD_INFRACAO`** (dicionário à parte), "
           "eliminando redundância." if fd_enq and fd_desc else
           "Há códigos com múltiplos textos → manter os três na base analítica; "
           "na modelável resumir via dicionário código→texto."),
        f"- **`NOME_VEICULO_MARCA` + `NOM_MODELO_VEICULO`**: ≈ {red['n_marca']:,} marcas e "
        f"≈ {red['n_modelo']:,} modelos (carteira aberta, grafias livres) → **resumir em "
        "`ESPECIE`/`TIPO_VEICULO`** (já padronizados) ou dicionário de marcas controlado "
        "antes de uso como categoria.",
        f"- **`NOM_MUNICIPIO`**: ≈ {red['n_municipio']:,} municípios distintos (texto) → "
        "**resumir em código IBGE** (ou `UF_INFRACAO`+código) para agregação/geolocalização "
        "estável.",
        "- **`HORA`**: detalhe fino já coberto por `MES`/`ANO`/`QUARTIL` → em bases "
        "modeláveis **resumir em turno/faixa horária** (MADRUGADA/MANHÃ/TARDE/NOITE).",
        "- **`NUMERO_AUTO`**: identificador (≈1 por linha) — **não resume**: é a chave do auto "
        "de infração; servir de join/contagem, não de categoria.",
        "",
        "## 5. Colunas que PRECISAM de resumo/atenção",
        "",
        "- **Lacunas de era** (null100% em eras inteiras — ausência legítima na fonte, "
        "não erro): `NUMERO_AUTO` (era1), `HORA` (era1–2 parcial), `TIPO_VEICULO` (era1–2), "
        "`QTD_INFRACOES` (era1–2), `COD_INFRACAO` (2007) — ver matriz §2. "
        "Em cruzamentos entre eras, usar apenas colunas com cobertura nas duas.",
        "- **Sentinelas de origem** (`'-1`, `-1`, `NA`…) convertidas para `NULL` "
        "(§3) — nunca imputar 0/'' nelas: o `NULL` é a marca de \"não informado\".",
        f"- **`MED_REALIZADA`/`MED_CONSIDERADA`**: "
        + (f"100% numéricas após vírgula→ponto (0 inválidas em "
           f"{red['med_nao_nulo']:,}/{red['medc_nao_nulo']:,}) → **podem virar DOUBLE** "
           "numa próxima etapa sem perda." if med_ok and medc_ok else
           "ainda com valores não numéricos misturados → **precisam de resumo/conversão** "
           "controlada (mapear unidades) antes de virar DOUBLE."),
        f"- **`UF_PLACA`/`UF_INFRACAO`**: apenas 27 UFs aceitas; inválidas viram `NULL` "
        f"({sum(m['uf_placa_invalida'] + m['uf_infracao_invalida'] for m in validacoes):,} "
        "células corrigidas no total).",
        "",
        "---",
        "Leitura: `read_parquet('dados_tratados/base_analitica/*.parquet')` (DuckDB/pandas).",
    ]

    out = TRATADOS / "resumo_colunas.md"
    out.write_text("\n".join(linhas_md) + "\n", encoding="utf-8")
    print(f"resumo -> {out}", flush=True)
    return out


def main() -> None:
    argv = sys.argv[1:]
    so_resumo = "--resumo" in argv
    anos = [int(a) for a in argv if a.isdigit()] or ([] if so_resumo else list(range(2007, 2026)))
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)
    if len(anos) == 19:
        VALIDACAO.unlink(missing_ok=True)  # recomeco completo

    con = duckdb.connect()
    con.execute("SET threads=2")
    con.execute("SET memory_limit='2400MB'")
    con.execute(f"SET temp_directory='{TMP_DIR}'")
    con.execute("SET preserve_insertion_order=false")

    validacoes: list[dict] = []
    t0 = time.time()
    for ano in anos:
        ta = time.time()
        m = tratar_ano(con, ano)
        validacoes.append(m)
        with VALIDACAO.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(m, ensure_ascii=False) + "\n")
        print(
            f"[{ano}] {m['linhas']:,} linhas -> {m['parquet_mb']} MB "
            f"({time.time() - ta:.0f}s) | uf_placa_inv={m['uf_placa_invalida']} "
            f"uf_infra_inv={m['uf_infracao_invalida']} especie_sent={m['especie_sentinela']} "
            f"data_ruim={m['data_ruim']} ano_div={m['ano_divergente']}",
            flush=True,
        )

    if so_resumo or len(anos) == 19:
        if so_resumo:
            validacoes = [
                json.loads(l)
                for l in VALIDACAO.read_text(encoding="utf-8").splitlines()
                if l.strip()
            ]
        # fase B so faz sentido com a base completa (19 no jsonl)
        fase_b(con, validacoes)
    else:
        print("parcial: pulando fase B (rode com --resumo apos os 19)", flush=True)

    if so_resumo:
        print("resumo ok", flush=True)
        return
    total = sum(m["linhas"] for m in validacoes)
    print(f"OK {len(anos)} anos, {total:,} linhas, {time.time() - t0:.0f}s", flush=True)
    if len(anos) == 19 and total != TOTAL_ESPERADO:
        raise SystemExit(f"TOTAL {total} != {TOTAL_ESPERADO}")


if __name__ == "__main__":
    main()

Overwriting scripts/04_tratar.py


**Reexecução (opcional):** a Fase 1 precisa rebaixar as fontes do Drive (ver
`dados_brutos/README.md`); a Fase 2 roda direto sobre os `.csv.gz` — cerca de 1 h:

In [8]:
if EXECUTAR_PIPELINE:
    import subprocess, sys
    subprocess.run([sys.executable, "scripts/04_tratar.py"], check=True)
else:
    print("EXECUTAR_PIPELINE=False — pulando reexecução; resultados já em dados_tratados/")

EXECUTAR_PIPELINE=False — pulando reexecução; resultados já em dados_tratados/


## 4. Validações executáveis

Rodem em segundos sobre os parquets gerados (se `dados_tratados/` não existir, rode a Fase 2 primeiro).

### 4.1 Contagem total e período

In [9]:
r = con.execute("""
SELECT count(*) total, count(DISTINCT ANO) anos,
       min(DATA_INFRACAO) dmin, max(DATA_INFRACAO) dmax
FROM read_parquet('dados_tratados/base_analitica/*.parquet')
""").fetchone()
print(f"Linhas: {r[0]:,} | Anos: {r[1]} | Periodo: {r[2]} a {r[3]}")
assert r[0] == 92_215_588, "total divergente"
print("OK: 92.215.588 linhas")

Linhas: 92,215,588 | Anos: 19 | Periodo: 2007-01-01 a 2025-12-31
OK: 92.215.588 linhas


### 4.2 `MES` (nome) e `QUARTIL` (4×3 meses) coerentes com `DATA_INFRACAO`

In [10]:
meses = [x[0] for x in con.execute("""
    SELECT MES FROM read_parquet('dados_tratados/base_analitica/*.parquet')
    GROUP BY MES ORDER BY MIN(DATA_INFRACAO)
""").fetchall()]
print("MES:", meses)

r = con.execute("""
SELECT count(*),
       count(DISTINCT MES),
       count(*) FILTER (WHERE MES IS NULL),
       count(*) FILTER (WHERE MES <> CASE EXTRACT(MONTH FROM DATA_INFRACAO)
            WHEN 1 THEN 'JANEIRO' WHEN 2 THEN 'FEVEREIRO' WHEN 3 THEN 'MARCO'
            WHEN 4 THEN 'ABRIL' WHEN 5 THEN 'MAIO' WHEN 6 THEN 'JUNHO'
            WHEN 7 THEN 'JULHO' WHEN 8 THEN 'AGOSTO' WHEN 9 THEN 'SETEMBRO'
            WHEN 10 THEN 'OUTUBRO' WHEN 11 THEN 'NOVEMBRO' WHEN 12 THEN 'DEZEMBRO' END),
       count(*) FILTER (WHERE QUARTIL <> CASE
            WHEN EXTRACT(MONTH FROM DATA_INFRACAO) <= 3 THEN 'Q1'
            WHEN EXTRACT(MONTH FROM DATA_INFRACAO) <= 6 THEN 'Q2'
            WHEN EXTRACT(MONTH FROM DATA_INFRACAO) <= 9 THEN 'Q3' ELSE 'Q4' END)
FROM read_parquet('dados_tratados/base_analitica/*.parquet')
""").fetchone()
print(f"total={r[0]:,} distinct_MES={r[1]} nulos_MES={r[2]} MES_fora={r[3]} QUARTIL_div={r[4]}")
assert r[1] == 12 and r[2] == 0 and r[3] == 0 and r[4] == 0
print("OK: MES = 12 nomes, QUARTIL = Q1..Q4, tudo coerente")

MES: ['JANEIRO', 'FEVEREIRO', 'MARCO', 'ABRIL', 'MAIO', 'JUNHO', 'JULHO', 'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']


total=92,215,588 distinct_MES=12 nulos_MES=0 MES_fora=0 QUARTIL_div=0
OK: MES = 12 nomes, QUARTIL = Q1..Q4, tudo coerente


### 4.3 Schema: 28 colunas, todas em MAIÚSCULOS

In [11]:
cols = con.execute("""
    DESCRIBE SELECT * FROM read_parquet('dados_tratados/base_analitica/infracoes_2007.parquet')
""").fetchall()
print(f"{len(cols)} colunas | todas MAIUSCULAS: {all(c[0] == c[0].upper() for c in cols)}")
for c in cols:
    print(f"  {c[0]:<24} {c[1]}")
assert len(cols) == 28 and all(c[0] == c[0].upper() for c in cols)

28 colunas | todas MAIUSCULAS: True
  NUMERO_AUTO              VARCHAR
  DATA_INFRACAO            DATE
  HORA                     TIME
  TIP_ABORDAGEM            VARCHAR
  IND_ASSINOU_AUTO         VARCHAR
  IND_VEICULO_ESTRANGEIRO  VARCHAR
  SENTIDO_TRAFEGO          VARCHAR
  UF_PLACA                 VARCHAR
  UF_INFRACAO              VARCHAR
  NUM_BR_INFRACAO          BIGINT
  NUM_KM_INFRACAO          DOUBLE
  NOM_MUNICIPIO            VARCHAR
  COD_INFRACAO             VARCHAR
  DESCRICAO_ABRVIADA       VARCHAR
  ENQUADRAMENTO            VARCHAR
  DATA_INICIO_VIGENCIA     DATE
  DATA_FIM_VIGENCIA        DATE
  MED_REALIZADA            VARCHAR
  MED_CONSIDERADA          VARCHAR
  EXC_VERIFICADO           VARCHAR
  ESPECIE                  VARCHAR
  NOME_VEICULO_MARCA       VARCHAR
  NOM_MODELO_VEICULO       VARCHAR
  TIPO_VEICULO             VARCHAR
  QTD_INFRACOES            BIGINT
  MES                      VARCHAR
  ANO                      SMALLINT
  QUARTIL                  VARCHA

### 4.4 Amostra: era1 (2010, lacunas legítimas) vs era3 (2024, completo)

In [12]:
for ano in (2010, 2024):
    row = con.execute(f"""
        SELECT DATA_INFRACAO, MES, ANO, QUARTIL, HORA, UF_INFRACAO,
               ESPECIE, TIPO_VEICULO, QTD_INFRACOES
        FROM read_parquet('dados_tratados/base_analitica/*.parquet')
        WHERE ANO = {ano} LIMIT 2
    """).fetchall()
    print(f"--- {ano} ---")
    for x in row:
        print("  ", x)

--- 2010 ---
   (datetime.date(2010, 1, 1), 'JANEIRO', 2010, 'Q1', None, 'RO', 'PASSAGEIRO', None, None)
   (datetime.date(2010, 1, 1), 'JANEIRO', 2010, 'Q1', None, 'RJ', 'PASSAGEIRO', None, None)
--- 2024 ---
   (datetime.date(2024, 1, 2), 'JANEIRO', 2024, 'Q1', datetime.time(2, 0), 'ES', 'MISTO', 'UTILITARIO', 1)
   (datetime.date(2024, 1, 2), 'JANEIRO', 2024, 'Q1', datetime.time(2, 0), 'ES', 'PASSAGEIRO', 'AUTOMOVEL', 1)


### 4.5 Nulos globais (resumo rápido das colunas)

In [13]:
cols = [c[0] for c in con.execute("""
    DESCRIBE SELECT * FROM read_parquet('dados_tratados/base_analitica/infracoes_2007.parquet')
""").fetchall()]
aggs = ", ".join(
    f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL) / count(*), 1) AS \"{c}\""
    for c in cols
)
row = con.execute(
    f"SELECT {aggs} FROM read_parquet('dados_tratados/base_analitica/*.parquet')"
).fetchone()
for c, v in sorted(zip(cols, row), key=lambda x: -x[1]):
    if v and v > 0:
        print(f"  {c:<24} {v}%")
print("(so colunas com nulos > 0; ver resumo_colunas.md para a matriz por era)")

  TIPO_VEICULO             68.6%
  DATA_FIM_VIGENCIA        64.8%
  QTD_INFRACOES            57.3%
  IND_ASSINOU_AUTO         56.5%
  NUMERO_AUTO              51.0%
  COD_INFRACAO             36.5%
  NOME_VEICULO_MARCA       31.8%
  MED_REALIZADA            29.6%
  HORA                     24.6%
  SENTIDO_TRAFEGO          23.7%
  NOM_MODELO_VEICULO       20.3%
  EXC_VERIFICADO           19.8%
  MED_CONSIDERADA          19.2%
  ESPECIE                  18.6%
  UF_PLACA                 15.6%
  DATA_INICIO_VIGENCIA     0.1%
(so colunas com nulos > 0; ver resumo_colunas.md para a matriz por era)


### 4.6 Relatórios gerados na Fase 2

In [14]:
for nome in ("dados_tratados/resumo_colunas.md", "dados_tratados/valores_rejeitados.md"):
    p = Path(nome)
    print("=" * 30, nome, "=" * 30)
    if p.exists():
        linhas = p.read_text(encoding="utf-8").splitlines()
        print("\n".join(linhas[:25]))
        print(f"... ({len(linhas)} linhas no total)")
    else:
        print("AUSENTE — rode: .venv/bin/python scripts/04_tratar.py")
    print()

============================== dados_tratados/resumo_colunas.md ==============================
# resumo_colunas — base analítica (dados_tratados)

- **Linhas:** 92,215,588 · **Colunas:** 28 · Gerado por `scripts/04_tratar.py` em 2026-11-25 14:43.
- **Fonte:** `dados_brutos/infracoes_2007..2025.csv.gz` (19 arquivos, schema canônico 25 col).
- Colunas de texto: sentinela → `NULL`; `TRIM` + `MAIÚSCULAS` + acentos→ASCII. Colunas derivadas: `MES` (nome do mês: JANEIRO…DEZEMBRO), `ANO`, `QUARTIL` (Q1=jan-mar, Q2=abr-jun, Q3=jul-set, Q4=out-dez).

## 1. Estatísticas globais

| # | Coluna | Tipo | Nulos | Nulos % | Distintos (aprox) |
|---|--------|------|------:|--------:|------------------:|
| 1 | `NUMERO_AUTO` | VARCHAR | 47,053,140 | 51.03% | 46,181,829 |
| 2 | `DATA_INFRACAO` | DATE | 0 | 0.00% | 6,969 |
| 3 | `HORA` | TIME | 22,679,289 | 24.59% | 85,475 |
| 4 | `TIP_ABORDAGEM` | VARCHAR | 0 | 0.00% | 2 |
| 5 | `IND_ASSINOU_AUTO` | VARCHAR | 52,097,346 | 56.50% | 2 |
| 6 | `IND_VEICULO_ES

## 5. Histórico de alterações

| # | Alteração | Onde |
|---|-----------|------|
| 1 | Fase 1 completa: RARs → unificação → normalização 25 col (92.215.588) | `extract_rar.js`, `01`, `02`, `03` · PR #1 |
| 2 | **jun/2024 inexistente** → `MESES_INEXISTENTES = {2024: [6]}` (INFO, não pendência) | `02` · commit `5c3caba` |
| 3 | Publicação de `dados_brutos/` no GitHub: gzip + volumes ≤95 MB (limite Git 100 MB) + `SHA256SUMS.txt` (uploads de Release bloqueados no sandbox) | PR #2 |
| 4 | Fallback `parallel=false` no `03` (reader do DuckDB recusa o 2023) | `03` |
| 5 | **Fase 2**: `04_tratar` → base analítica **28 colunas** Parquet; sentinela/UF/acentos padronizados | PR #3 |
| 6 | **`QUARTIL` = 4 períodos de 3 meses** (Q1–Q4) confirmada pelo usuário | `04` |
| 7 | **`MES`: número → nome do mês** (JANEIRO…DEZEMBRO) — migração dos 19 parquets em 74 s | `04` · commit `ff7746c` |
| 8 | **`valores_rejeitados.md`**: lista completa do que virou `NULL` (108 valores de UF; 2 sentinelas de `ESPECIE`) | varredura da fonte |

## Referências

- `dados_brutos/README.md` — 19 CSVs, checksums, regeneração (Release [`dados-brutos`](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/releases/tag/dados-brutos))
- `dados_tratados/README.md` · `resumo_colunas.md` · `valores_rejeitados.md`
- `logs/ingestao_multas.md` — log completo das execuções
- PRs: [#1](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/pull/1) · [#2](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/pull/2) · [#3](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/pull/3)